In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             precision_recall_curve, roc_curve, auc)

# Load the dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Note: In sklearn, 0 = Malignant, 1 = Benign.
# For medical analysis, we typically treat "Malignant" (Cancer) as the Positive case (1).
# We invert y so that 1 = Malignant and 0 = Benign.
y = (y == 0).astype(int)
target_names = ['Benign', 'Malignant']

# Split dataset into training and testing sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("Data loaded and split successfully.")
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

In [ ]:

# Cell 2: Experiment 1 - Naïve Bayes Model & Metrics
print("--- EXPERIMENT 1: NAÏVE BAYES ---")

# Train Naïve Bayes classifier
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

# Predict class labels
y_pred_nb = nb_model.predict(X_test)
y_prob_nb = nb_model.predict_proba(X_test)[:, 1]  # Probabilities for curve plotting

# Evaluate metrics
acc_nb = accuracy_score(y_test, y_pred_nb)
cm_nb = confusion_matrix(y_test, y_pred_nb)

print(f"Naïve Bayes Accuracy: {acc_nb:.4f}")
print("\nConfusion Matrix:\n", cm_nb)
print("\nClassification Report:\n", classification_report(y_test, y_pred_nb, target_names=target_names))

In [ ]:


# Cell 3: Experiment 1 - Precision-Recall Curve
precision_nb, recall_nb, _ = precision_recall_curve(y_test, y_prob_nb)

plt.figure(figsize=(8, 6))
plt.plot(recall_nb, precision_nb, color='blue', lw=2, label='Naïve Bayes')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Experiment 1: Precision-Recall Curve (Naïve Bayes)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:

# Cell 4: Experiment 2 - Decision Tree Model & Metrics
print("--- EXPERIMENT 2: DECISION TREE ---")

# Train Decision Tree classifier
# We set max_depth=4 to prevent extreme overfitting and make the tree visualization readable
dt_model = DecisionTreeClassifier(random_state=42, max_depth=4)
dt_model.fit(X_train, y_train)

# Predict class labels
y_pred_dt = dt_model.predict(X_test)
y_prob_dt = dt_model.predict_proba(X_test)[:, 1]

# Evaluate metrics
acc_dt = accuracy_score(y_test, y_pred_dt)
cm_dt = confusion_matrix(y_test, y_pred_dt)

print(f"Decision Tree Accuracy: {acc_dt:.4f}")
print("\nConfusion Matrix:\n", cm_dt)
print("\nClassification Report:\n", classification_report(y_test, y_pred_dt, target_names=target_names))

In [ ]:

# Cell 5: Experiment 2 - Precision-Recall Curve
precision_dt, recall_dt, _ = precision_recall_curve(y_test, y_prob_dt)

plt.figure(figsize=(8, 6))
plt.plot(recall_dt, precision_dt, color='green', lw=2, label='Decision Tree')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Experiment 2: Precision-Recall Curve (Decision Tree)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:

# Cell 6: Experiment 2 - Visualize the Decision Tree
plt.figure(figsize=(25, 12))
plot_tree(dt_model, feature_names=data.feature_names,
          class_names=target_names, filled=True, fontsize=10, rounded=True)
plt.title("Experiment 2: Decision Tree Visualization")
plt.show()
     

In [ ]:

# Cell 7: Experiment 3 - Accuracy Comparison Bar Chart
train_acc_nb = accuracy_score(y_train, nb_model.predict(X_train))
train_acc_dt = accuracy_score(y_train, dt_model.predict(X_train))

labels = ['Naïve Bayes', 'Decision Tree']
train_accs = [train_acc_nb, train_acc_dt]
test_accs = [acc_nb, acc_dt]

x = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(8, 6))
plt.bar(x - width/2, train_accs, width, label='Training Accuracy', color='skyblue')
plt.bar(x + width/2, test_accs, width, label='Testing Accuracy', color='orange')
plt.xlabel('Models')
plt.ylabel('Accuracy')
plt.title('Experiment 3: Training vs Testing Accuracy Comparison')
plt.xticks(x, labels)
plt.legend()
plt.ylim(0.85, 1.01) # Set limit to see the difference clearly
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:


# Cell 8: Experiment 3 - ROC Curve Comparison
fpr_nb, tpr_nb, _ = roc_curve(y_test, y_prob_nb)
roc_auc_nb = auc(fpr_nb, tpr_nb)

fpr_dt, tpr_dt, _ = roc_curve(y_test, y_prob_dt)
roc_auc_dt = auc(fpr_dt, tpr_dt)

plt.figure(figsize=(8, 6))
plt.plot(fpr_nb, tpr_nb, color='blue', lw=2, label=f'Naïve Bayes (AUC = {roc_auc_nb:.2f})')
plt.plot(fpr_dt, tpr_dt, color='green', lw=2, label=f'Decision Tree (AUC = {roc_auc_dt:.2f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('Experiment 3: ROC Curve Comparison')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

In [ ]:

# Cell 9: Experiment 3 - Confusion Matrix Heatmaps
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Naïve Bayes Heatmap
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=target_names, yticklabels=target_names)
axes[0].set_title('Naïve Bayes Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Decision Tree Heatmap
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=target_names, yticklabels=target_names)
axes[1].set_title('Decision Tree Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_recall_curve, roc_curve, auc

# Load Data
data = load_breast_cancer()
X, y = pd.DataFrame(data.data, columns=data.feature_names), (data.target == 0).astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
names = ['Benign', 'Malignant']

# --- Exp 1: Naïve Bayes ---
nb = GaussianNB().fit(X_train, y_train)
prob_nb = nb.predict_proba(X_test)[:, 1]
print(f"NB Acc: {accuracy_score(y_test, nb.predict(X_test)):.4f}\nCM:\n{confusion_matrix(y_test, nb.predict(X_test))}")
print("Report:\n", classification_report(y_test, nb.predict(X_test), target_names=names))

# Plot NB Precision-Recall
p, r, _ = precision_recall_curve(y_test, prob_nb)
plt.plot(r, p, label='NB'); plt.title('Exp 1: NB Precision-Recall'); plt.show()

# --- Exp 2: Decision Tree ---
dt = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_train, y_train)
prob_dt = dt.predict_proba(X_test)[:, 1]
print(f"DT Acc: {accuracy_score(y_test, dt.predict(X_test)):.4f}\nCM:\n{confusion_matrix(y_test, dt.predict(X_test))}")
print("Report:\n", classification_report(y_test, dt.predict(X_test), target_names=names))

# Plot DT Precision-Recall & Tree
p_dt, r_dt, _ = precision_recall_curve(y_test, prob_dt)
plt.plot(r_dt, p_dt, color='g', label='DT'); plt.title('Exp 2: DT Precision-Recall'); plt.show()
plt.figure(figsize=(15,8)); plot_tree(dt, feature_names=data.feature_names, class_names=names, filled=True); plt.show()

# --- Exp 3: Comparison ---
# 1. Accuracy Bar Chart
accs = [accuracy_score(y_train, m.predict(X_train)) for m in [nb, dt]]
test_accs = [accuracy_score(y_test, m.predict(X_test)) for m in [nb, dt]]
plt.bar(['NB','DT'], accs, width=0.4, label='Train'); plt.bar(['NB','DT'], test_accs, width=0.4, label='Test', align='edge')
plt.legend(); plt.title('Exp 3: Accuracy Comparison'); plt.show()

# 2. ROC Curve
fpr_nb, tpr_nb, _ = roc_curve(y_test, prob_nb)
fpr_dt, tpr_dt, _ = roc_curve(y_test, prob_dt)
plt.plot(fpr_nb, tpr_nb, label=f'NB (AUC={auc(fpr_nb, tpr_nb):.2f})')
plt.plot(fpr_dt, tpr_dt, label=f'DT (AUC={auc(fpr_dt, tpr_dt):.2f})')
plt.plot([0,1],[0,1],'k--'); plt.legend(); plt.title('Exp 3: ROC Comparison'); plt.show()

# 3. Heatmaps
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
sns.heatmap(confusion_matrix(y_test, nb.predict(X_test)), annot=True, fmt='d', cmap='Blues', ax=ax[0])
sns.heatmap(confusion_matrix(y_test, dt.predict(X_test)), annot=True, fmt='d', cmap='Greens', ax=ax[1])
ax[0].set_title('NB CM'); ax[1].set_title('DT CM'); plt.show()